# Step 1: Gradient Descent

Bu bölümde Türkçe isimlerle bir bigram modeli oluşturacağım.
Sayım yapmak yerine modelin ağırlıklarını gradient descent eğiteceğim.

## 1.1 Aynı Başlangıç Noktası

Önceki bölümdeki veri kümesini ve karakter tabanlı tokenizer'ı
kullanacağım.

## 1.1 Aynı Başlangıç Noktası

Önceki bölümdeki 1100 Türkçe isimlik veri kümesini ve aynı tokenizer'ı
kullanıyorum. Vocabulary, 29 karakter ve BOS tokenıyla toplam 30 tokendan
oluşuyor.

Model yine mevcut tokenı alıp sonraki tokenların olasılıklarını üretecek.
Değişen kısım, bu olasılıkların hesaplanması ve öğrenme yöntemi olacak.

In [1]:
import random
import math

docs = [
    l.strip().replace(" ", "").replace("İ", "i").replace("I", "ı").lower()
    for l in open("../data/raw_names.txt", encoding="utf-8")
        .read().strip().split("\n")
    if l.strip()
]

random.shuffle(docs)

uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

print("İsim sayısı:", len(docs))
print("Vocabulary boyutu:", vocab_size)

İsim sayısı: 1100
Vocabulary boyutu: 30


## 1.2 Parametreler

Modelin parametreleri üç ağırlık matrisinden oluşuyor:

wte: Her tokenı 16 sayılık bir embedding vektörüyle temsil eder.
mlp_fc1: Bu 16 sayıyı 64 sayılık bir ara temsile dönüştürür.
mlp_fc2: Ara temsilden, 30 olası sonraki token için skor üretir.

Parametreler küçük rastgele değerlerle başlatılır.
Eğitim sırasında bu değerler güncellenecektir.

In [2]:
n_embd = 16

matrix = lambda nout, nin: [
    [random.gauss(0, 0.08) for _ in range(nin)]
    for _ in range(nout)
]

state_dict = {
    'wte': matrix(vocab_size, n_embd),
    'mlp_fc1': matrix(4 * n_embd, n_embd),
    'mlp_fc2': matrix(vocab_size, 4 * n_embd),
}

params = [
    (row, j)
    for mat in state_dict.values()
    for row in mat
    for j in range(len(row))
]

In [3]:
for name, mat in state_dict.items():
    print(f"{name}: {len(mat)} × {len(mat[0])}")

print("Toplam parametre:", len(params))

wte: 30 × 16
mlp_fc1: 64 × 16
mlp_fc2: 30 × 64
Toplam parametre: 3424


## 1.3 Linear ve Softmax

Linear fonksiyonu, girdi vektörünü ağırlık matrisiyle çarparak yeni
bir vektör oluşturur. Her çıktı, girdilerin ağırlıklı toplamıdır.

Softmax fonksiyonu, logits adı verilen ham skorları toplamı 1 olan
olasılıklara dönüştürür.

In [4]:
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]


def softmax(logits):
    max_val = max(logits)
    exps = [math.exp(v - max_val) for v in logits]
    total = sum(exps)
    return [e / total for e in exps]

In [5]:
x = [2.0, -1.0, 0.5]

w = [
    [0.3, 0.7, -0.2],
    [-0.5, 0.1, 0.8],
]

print("Linear çıktısı:", linear(x, w))

probs = softmax([2, 1, 0])

print("Softmax çıktısı:", probs)
print("Olasılık toplamı:", sum(probs))

Linear çıktısı: [-0.19999999999999998, -0.7]
Softmax çıktısı: [0.6652409557748218, 0.24472847105479764, 0.09003057317038046]
Olasılık toplamı: 0.9999999999999999


## 1.4 MLP Modeli
MLP (Multi-Layer Perceptron), çok katmanlı algılayıcı anlamına gelir.
Model, mevcut tokenın ID'sini alarak sonraki 30 token için ham skorlar üretir.
ReLU, modele doğrusal olmayan bir işlem kazandırır. Logits değerleri
softmax ile olasılıklara dönüştürülür.

Model hâlâ yalnızca mevcut tokenı kullanır; önceki tokenları hatırlamaz.
Bu aşamada ağırlıklar güncellenmez, yalnızca tahmin hesaplanır.

In [6]:
def mlp(token_id):
    x= state_dict['wte'][token_id]
    x=linear(x,state_dict['mlp_fc1'])
    x=[max(0,xi)for xi in x]#relu
    logits = linear(x,state_dict['mlp_fc2'])
    return logits

In [7]:
token_id = uchars.index("a")

logits = mlp(token_id)
probs = softmax(logits)

print("Skor sayısı:", len(logits))
print("Olasılık sayısı:", len(probs))
print("Olasılıkların toplamı:", sum(probs))

Skor sayısı: 30
Olasılık sayısı: 30
Olasılıkların toplamı: 0.9999999999999999
